# SO101 Drop Quality ML Model

This notebook builds a binary classifier to predict **drop quality (Good vs Bad)** on the SO101 robot arm using joint telemetry data from `robot_demo_catalog.robot_demo.so101_snapshots`.

## Problem Definition
During pick-and-place cycles, the robot arm occasionally exhibits an anomalous **110° extra rotation** instead of the normal **100° rotation** at the drop phase. These bad drops risk misplacing or damaging objects. We label each cycle as:
- **Good (0)**: Normal 100° rotation cycle
- **Bad (1)**: Anomalous 110° extra rotation cycle

## Dataset
- **93 total cycles**: 62 Good, 31 Bad (~33% bad rate)
- **6 joints**: shoulder_pan, shoulder_lift, elbow_flex, wrist_flex, wrist_roll, gripper
- **Telemetry per joint**: position, speed, load, tracking_error, temperature, current
- **11 cycle phases**: raise gripper → open gripper → rotate to pick → reach table → hold for block → close gripper → raise to carry → rotate to drop → reach drop → open gripper drop → raise clear

## Approach
1. Parse JSON payloads and engineer 66 aggregate features per cycle
2. Train Random Forest and Gradient Boosted Tree classifiers with 5-fold stratified cross-validation
3. Evaluate with accuracy, F1, precision, recall, and AUC-ROC
4. Analyze feature importances to understand which joint behaviors drive bad drops

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *
import pandas as pd

# -------------------------------------------------------------------
# 1. Load raw snapshots and parse the JSON payload
# -------------------------------------------------------------------
raw_df = spark.table("robot_demo_catalog.robot_demo.so101_snapshots")

# Parse payload JSON — structure: {arms: [{serial, joints: [...]}], cycling, cycle_phase, ...}
# Joints are nested under arms[0].joints, each with: position, present_speed, load,
# tracking_error, temperature, current
parsed = raw_df.select(
    F.col("event_time"),
    F.get_json_object(F.col("payload"), "$.arms[0].serial").alias("arm_serial"),
    F.get_json_object(F.col("payload"), "$.cycling").cast("boolean").alias("cycling"),
    F.get_json_object(F.col("payload"), "$.cycle_index").cast("int").alias("cycle_index"),
    F.get_json_object(F.col("payload"), "$.cycle_phase").alias("cycle_phase"),
    # Joint arrays — 6 joints: shoulder_pan[0], shoulder_lift[1], elbow_flex[2],
    #                           wrist_flex[3], wrist_roll[4], gripper[5]
    F.from_json(
        F.get_json_object(F.col("payload"), "$.arms[0].joints"),
        ArrayType(StructType([
            StructField("position", DoubleType()),
            StructField("present_speed", DoubleType()),
            StructField("load", DoubleType()),
            StructField("tracking_error", DoubleType()),
            StructField("temperature", DoubleType()),
            StructField("current", DoubleType()),
        ]))
    ).alias("joints")
).filter("cycling = true AND cycle_index > 0")

# -------------------------------------------------------------------
# 2. Flatten joints into individual columns per joint index
# -------------------------------------------------------------------
joint_names = ["shoulder_pan", "shoulder_lift", "elbow_flex", "wrist_flex", "wrist_roll", "gripper"]
metrics = ["position", "present_speed", "load", "tracking_error", "temperature", "current"]

flat_cols = [F.col("arm_serial"), F.col("cycle_index"), F.col("cycle_phase")]
for i, jname in enumerate(joint_names):
    for m in metrics:
        flat_cols.append(F.col("joints")[i][m].cast("double").alias(f"{jname}_{m}"))

flat_df = parsed.select(*flat_cols)

# -------------------------------------------------------------------
# 3. Build aggregate features per cycle
# -------------------------------------------------------------------
# Label: is_bad = 1 if any phase in cycle contains '110 deg extra'
agg_exprs = [
    F.max(F.when(F.col("cycle_phase").like("%110 deg extra%"), F.lit(1)).otherwise(F.lit(0))).alias("is_bad")
]

for jname in joint_names:
    # Position: mean, std, range
    pos = f"{jname}_position"
    agg_exprs.append(F.mean(pos).alias(f"{jname}_pos_mean"))
    agg_exprs.append(F.stddev(pos).alias(f"{jname}_pos_std"))
    agg_exprs.append((F.max(pos) - F.min(pos)).alias(f"{jname}_pos_range"))
    # Speed (absolute): mean, max
    spd = f"{jname}_present_speed"
    agg_exprs.append(F.mean(F.abs(F.col(spd))).alias(f"{jname}_speed_mean"))
    agg_exprs.append(F.max(F.abs(F.col(spd))).alias(f"{jname}_speed_max"))
    # Load (absolute): mean, max
    ld = f"{jname}_load"
    agg_exprs.append(F.mean(F.abs(F.col(ld))).alias(f"{jname}_load_mean"))
    agg_exprs.append(F.max(F.abs(F.col(ld))).alias(f"{jname}_load_max"))
    # Tracking error (absolute): mean, max
    te = f"{jname}_tracking_error"
    agg_exprs.append(F.mean(F.abs(F.col(te))).alias(f"{jname}_te_mean"))
    agg_exprs.append(F.max(F.abs(F.col(te))).alias(f"{jname}_te_max"))
    # Temperature: mean
    agg_exprs.append(F.mean(f"{jname}_temperature").alias(f"{jname}_temp_mean"))
    # Current: mean
    agg_exprs.append(F.mean(f"{jname}_current").alias(f"{jname}_current_mean"))

cycle_features = flat_df.groupBy("arm_serial", "cycle_index").agg(*agg_exprs)

# -------------------------------------------------------------------
# 4. Convert to pandas for sklearn
# -------------------------------------------------------------------
pdf = cycle_features.toPandas()

# Separate features and label
label_col = "is_bad"
meta_cols = ["arm_serial", "cycle_index", "is_bad"]
feature_cols = [c for c in pdf.columns if c not in meta_cols]

# Fill any nulls from stddev (single-observation groups) with 0
pdf[feature_cols] = pdf[feature_cols].fillna(0)

X = pdf[feature_cols]
y = pdf[label_col]

print(f"Dataset shape: {X.shape[0]} cycles x {X.shape[1]} features")
print(f"Label distribution:\n  Good (0): {(y == 0).sum()}\n  Bad  (1): {(y == 1).sum()}")
print(f"\nFeature columns ({len(feature_cols)}):\n{feature_cols}")

In [0]:
import mlflow
import mlflow.sklearn
import numpy as np
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix, classification_report
)
from mlflow.models import infer_signature

# -------------------------------------------------------------------
# MLflow experiment setup
# -------------------------------------------------------------------
experiment_path = "/Users/noel.hoxie@databricks.com/SO101_Drop_Quality_ML"
mlflow.set_experiment(experiment_path)

# -------------------------------------------------------------------
# Model definitions
# -------------------------------------------------------------------
models = {
    "RandomForest": RandomForestClassifier(
        n_estimators=200, max_depth=10, min_samples_leaf=2,
        class_weight="balanced", random_state=42, n_jobs=-1
    ),
    "GradientBoostedTrees": GradientBoostingClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.1,
        min_samples_leaf=3, random_state=42
    ),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

# -------------------------------------------------------------------
# Train, evaluate, and log each model
# -------------------------------------------------------------------
for model_name, estimator in models.items():
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", estimator),
    ])

    # Cross-validated predictions (out-of-fold)
    y_pred = cross_val_predict(pipe, X, y, cv=cv, method="predict")
    y_proba = cross_val_predict(pipe, X, y, cv=cv, method="predict_proba")[:, 1]

    # Metrics
    acc = accuracy_score(y, y_pred)
    f1 = f1_score(y, y_pred)
    prec = precision_score(y, y_pred)
    rec = recall_score(y, y_pred)
    auc = roc_auc_score(y, y_proba)
    cm = confusion_matrix(y, y_pred)

    results[model_name] = {
        "accuracy": acc, "f1": f1, "precision": prec,
        "recall": rec, "auc_roc": auc, "confusion_matrix": cm,
    }

    # Fit final model on all data and log to MLflow
    pipe.fit(X, y)

    with mlflow.start_run(run_name=model_name):
        mlflow.log_params({
            "model_type": model_name,
            "n_features": X.shape[1],
            "n_samples": X.shape[0],
            "cv_folds": 5,
        })
        mlflow.log_metrics({
            "accuracy": round(acc, 4),
            "f1_score": round(f1, 4),
            "precision": round(prec, 4),
            "recall": round(rec, 4),
            "auc_roc": round(auc, 4),
        })

        signature = infer_signature(X.head(5), pipe.predict(X.head(5)))
        mlflow.sklearn.log_model(
            pipe, name="model",
            signature=signature,
            input_example=X.head(3),
        )

    # Store the fitted pipeline for feature importance extraction
    if model_name == "RandomForest":
        rf_pipeline = pipe

# -------------------------------------------------------------------
# Print comparison
# -------------------------------------------------------------------
print("=" * 60)
print("MODEL COMPARISON — 5-Fold Stratified Cross-Validation")
print("=" * 60)
for name, r in results.items():
    print(f"\n{name}:")
    print(f"  Accuracy : {r['accuracy']:.3f}")
    print(f"  F1 Score : {r['f1']:.3f}")
    print(f"  Precision: {r['precision']:.3f}")
    print(f"  Recall   : {r['recall']:.3f}")
    print(f"  AUC-ROC  : {r['auc_roc']:.3f}")
    print(f"  Confusion Matrix:\n    TN={r['confusion_matrix'][0,0]}  FP={r['confusion_matrix'][0,1]}")
    print(f"    FN={r['confusion_matrix'][1,0]}  TP={r['confusion_matrix'][1,1]}")

best = max(results, key=lambda k: results[k]["f1"])
print(f"\n>>> Best model: {best} (F1={results[best]['f1']:.3f}, AUC={results[best]['auc_roc']:.3f})")

In [0]:
import matplotlib.pyplot as plt

# Extract feature importances from the Random Forest within the pipeline
rf_model = rf_pipeline.named_steps["clf"]
importances = rf_model.feature_importances_

# Build a sorted DataFrame of importances
feat_imp_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": importances
}).sort_values("importance", ascending=False)

top15 = feat_imp_df.head(15)

# Display table
print("Top 15 Feature Importances (Random Forest)")
print("-" * 45)
for _, row in top15.iterrows():
    print(f"  {row['feature']:35s} {row['importance']:.4f}")

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top15["feature"][::-1], top15["importance"][::-1], color="steelblue")
ax.set_xlabel("Importance")
ax.set_title("Top 15 Feature Importances — Random Forest")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

## Results Summary

### Recommended Model: Random Forest
| Metric | Random Forest | Gradient Boosted Trees |
| --- | --- | --- |
| Accuracy | **96.8%** | 96.8% |
| F1 Score | 0.949 | **0.951** |
| AUC-ROC | **0.997** | 0.986 |
| Precision | **1.000** | 0.967 |
| Recall | 0.903 | **0.935** |

### Key Findings
- **Random Forest** is the recommended model with 96.8% accuracy, near-perfect **AUC of 0.997**, and **perfect precision** (zero false positives)
- While GBT edges ahead on F1 (0.951 vs 0.949), RF’s zero false positives make it the safer choice for quality-sensitive applications
- The RF model **perfectly identifies all good cycles** (zero false positives) — no good drops are misclassified as bad
- Only **3 out of 31 bad cycles are missed** (false negatives), giving 90.3% recall on the bad class

### Top Predictive Features
The most important features are dominated by the **shoulder_pan joint**:
1. **shoulder_pan position range** (0.207) — bad drops show the shoulder pan sweeping a wider arc due to the extra 110° rotation
2. **shoulder_pan current mean** (0.125) — the extra rotation draws measurably more motor current
3. **shoulder_pan position std** (0.119) — higher variability in shoulder pan position during anomalous cycles
4. **shoulder_pan load mean** (0.089) — the extra rotation creates higher mechanical load

This aligns with the physical mechanism: the 110° extra rotation anomaly directly manifests as abnormal shoulder pan movement and increased motor effort, making these features strong discriminators for drop quality prediction.